# Lab 4 v4 - Vì sao cần model AI trong IoT forecasting?

## Mục tiêu notebook
Notebook này giúp sinh viên hiểu trước khi train model:
- Forecasting khác anomaly detection ở đâu.
- Input `X` và target tương lai `y` được tạo như thế nào.
- Vì sao cần baseline trước khi dùng model AI.
- Vì sao time-series không được random split.

**Câu hỏi:** Nếu chỉ dùng ngưỡng cứng, hệ thống có dự báo được điều sắp xảy ra không?

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
print(ROOT)

In [ ]:
import pandas as pd
from config import DATASET_CONFIGS
from prepare_datasets import prepare_appliances, prepare_co2
prepare_appliances(); prepare_co2()
for key, cfg in DATASET_CONFIGS.items():
    df = pd.read_csv(cfg['processed_file'])
    print(key, df.shape, 'target=', cfg['target'], 'horizon_minutes=', cfg['horizon_minutes'])
    display(df.head())

## Từ time-series sang supervised learning
Với mỗi dòng tại thời điểm `t`, ta tạo feature từ hiện tại và quá khứ gần. Target là giá trị tại `t + horizon`.

**Câu hỏi:** Nếu horizon là 30 phút, model đang dự báo giá trị nào? Có được dùng dữ liệu sau thời điểm `t` làm feature không?

In [ ]:
from feature_engineering import make_supervised_forecasting_frame
cfg = DATASET_CONFIGS['appliances']
df = pd.read_csv(cfg['processed_file'])
supervised, feature_cols = make_supervised_forecasting_frame(df, cfg['timestamp'], cfg['target'], cfg['horizon_steps'], cfg['lags'], cfg['rolling_windows'])
print('feature count:', len(feature_cols))
display(supervised[[cfg['timestamp'], f"{cfg['target']}_current", f"{cfg['target']}_lag_1", f"{cfg['target']}_rolling_mean_{cfg['rolling_windows'][0]}", 'target_future']].head(10))

## Câu hỏi suy nghĩ
1. Lag feature giúp model nhìn thấy thông tin gì?
2. Rolling mean khác Last Value ở điểm nào?
3. Vì sao `target_future = target.shift(-horizon)` có thể gây nhầm nếu không giải thích kỹ?